# Exploratory Data Analysis - Cervical Cancer Risk Factors

This notebook performs exploratory data analysis on the cervical cancer risk factors dataset to understand the data distribution, missing values, correlations, and key patterns.

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully.')

In [ ]:
# Load the dataset
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "kag_risk_factors_cervical_cancer.csv"
df = pd.read_csv(DATA_PATH)
# Replace '?' with NaN
df.replace('?', np.nan, inplace=True)

print('Dataset loaded successfully.')

## 1. Basic Dataset Information

In [ ]:
# Display shape
print(f'Dataset Shape: {df.shape}')
print(f'Number of Rows: {df.shape[0]}')
print(f'Number of Columns: {df.shape[1]}')

In [ ]:
# Display first 10 rows
df.head(10)

In [ ]:
# Display column names
print('Column Names:')
for i, col in enumerate(df.columns, 1):
    print(f'{i}. {col}')

In [ ]:
# Data types and info
df.info()

In [ ]:
# Statistical summary
df.describe()

## 2. Missing Value Analysis

In [ ]:
# Count missing values per column
missing_values = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing_values,
    'Missing Percentage': missing_percent
}).sort_values(by='Missing Count', ascending=False)

print('Missing Values per Column:')
missing_df[missing_df['Missing Count'] > 0]

In [ ]:
# Visualize missing values
plt.figure(figsize=(14, 8))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis', yticklabels=False)
plt.title('Missing Values Heatmap', fontsize=16)
plt.xlabel('Columns')
plt.ylabel('Rows')
plt.tight_layout()
plt.show()

In [ ]:
# Bar plot of missing values
missing_vals = missing_df[missing_df['Missing Count'] > 0]
plt.figure(figsize=(12, 6))
plt.barh(missing_vals.index, missing_vals['Missing Count'], color='coral')
plt.xlabel('Missing Count')
plt.title('Missing Values by Column', fontsize=16)
plt.tight_layout()
plt.show()

## 3. Univariate Analysis

In [ ]:
# Convert columns to numeric for analysis
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print('Columns converted to numeric.')

In [ ]:
# Distribution of all numeric features
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
n_cols = len(numeric_cols)
n_rows = int(np.ceil(n_cols / 5))

plt.figure(figsize=(20, 5 * n_rows))
for i, col in enumerate(numeric_cols):
    plt.subplot(n_rows, 5, i + 1)
    sns.histplot(df[col].dropna(), bins=30, kde=True, color='steelblue')
    plt.title(f'Distribution of {col}', fontsize=10)
    plt.xlabel('')
    plt.xticks(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Age distribution
plt.figure(figsize=(10, 5))
sns.histplot(df['Age'].dropna(), bins=30, kde=True, color='steelblue')
plt.title('Age Distribution', fontsize=16)
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Box plots for key numeric features
key_features = ['Age', 'Number of sexual partners', 'First sexual intercourse', 
                'Num of pregnancies', 'Smokes (years)', 'Smokes (packs/year)']

plt.figure(figsize=(15, 10))
for i, col in enumerate(key_features, 1):
    plt.subplot(2, 3, i)
    sns.boxplot(y=df[col], color='lightblue')
    plt.title(f'Box Plot of {col}', fontsize=12)
plt.tight_layout()
plt.show()

## 4. Target Variable Analysis

In [ ]:
# Target variable: Biopsy
target = 'Biopsy'
print('Target Variable: Biopsy (0 = Negative, 1 = Positive)')
print(f'\nValue counts:\n{df[target].value_counts()}')
print(f'\nPercentage:\n{df[target].value_counts(normalize=True) * 100}')

In [ ]:
# Visualize target distribution
plt.figure(figsize=(8, 5))
counts = df[target].value_counts()
bars = plt.bar(counts.index, counts.values, color=['skyblue', 'salmon'], edgecolor='black', width=0.6)
plt.title('Biopsy Result Distribution', fontsize=16)
plt.xlabel('Biopsy Result (0 = Negative, 1 = Positive)', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks([0, 1], ['Negative (0)', 'Positive (1)'])

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{int(height)}', ha='center', va='bottom', fontsize=12)
plt.show()

In [ ]:
# Also check other diagnosis columns
dx_cols = ['Dx:Cancer', 'Dx:CIN', 'Dx:HPV', 'Dx', 'Hinselmann', 'Schiller', 'Citology', 'Biopsy']
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(dx_cols):
    if col in df.columns:
        counts = df[col].value_counts()
        axes[i].bar(counts.index, counts.values, color=['lightgreen', 'tomato'], edgecolor='black')
        axes[i].set_title(f'{col} Distribution', fontsize=12)
        axes[i].set_xlabel('Result')
        axes[i].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 5. Bivariate Analysis

In [ ]:
# Correlation heatmap
plt.figure(figsize=(18, 14))
corr_matrix = df.corr()
sns.heatmap(corr_matrix, cmap='coolwarm', annot=False, center=0, 
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Heatmap', fontsize=18)
plt.tight_layout()
plt.show()

In [ ]:
# Top correlations with target (Biopsy)
target_corr = corr_matrix['Biopsy'].drop('Biopsy').sort_values(ascending=False)

plt.figure(figsize=(10, 8))
colors = ['green' if v > 0 else 'red' for v in target_corr.values]
plt.barh(target_corr.index, target_corr.values, color=colors, edgecolor='black')
plt.title('Feature Correlations with Biopsy (Target)', fontsize=16)
plt.xlabel('Correlation Coefficient')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Age vs Biopsy
plt.figure(figsize=(10, 5))
sns.boxplot(x='Biopsy', y='Age', data=df, palette=['skyblue', 'salmon'])
plt.title('Age Distribution by Biopsy Result', fontsize=16)
plt.xlabel('Biopsy Result')
plt.ylabel('Age')
plt.show()

In [ ]:
# Number of sexual partners vs Biopsy
plt.figure(figsize=(10, 5))
sns.boxplot(x='Biopsy', y='Number of sexual partners', data=df, palette=['skyblue', 'salmon'])
plt.title('Number of Sexual Partners by Biopsy Result', fontsize=16)
plt.xlabel('Biopsy Result')
plt.ylabel('Number of Sexual Partners')
plt.show()

In [ ]:
# Num of pregnancies vs Biopsy
plt.figure(figsize=(10, 5))
sns.boxplot(x='Biopsy', y='Num of pregnancies', data=df, palette=['skyblue', 'salmon'])
plt.title('Number of Pregnancies by Biopsy Result', fontsize=16)
plt.xlabel('Biopsy Result')
plt.ylabel('Number of Pregnancies')
plt.show()

In [ ]:
# Smoking years vs Biopsy
plt.figure(figsize=(10, 5))
sns.boxplot(x='Biopsy', y='Smokes (years)', data=df, palette=['skyblue', 'salmon'])
plt.title('Smoking Years by Biopsy Result', fontsize=16)
plt.xlabel('Biopsy Result')
plt.ylabel('Smokes (years)')
plt.show()

In [ ]:
# Pairplot of selected features colored by Biopsy
selected_features = ['Age', 'Number of sexual partners', 'Num of pregnancies', 
                     'Smokes (years)', 'Hormonal Contraceptives (years)', 'Biopsy']

sns.pairplot(df[selected_features].dropna(), hue='Biopsy', palette=['skyblue', 'salmon'], diag_kind='hist')
plt.suptitle('Pairplot of Selected Features by Biopsy Result', y=1.02, fontsize=16)
plt.show()

## 6. Outlier Detection

In [ ]:
# Detect outliers using IQR method
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return len(outliers), lower_bound, upper_bound

print('Outlier Detection using IQR Method:')
print('=' * 60)
for col in ['Age', 'Number of sexual partners', 'First sexual intercourse', 
            'Num of pregnancies', 'Smokes (years)', 'Smokes (packs/year)']:
    count, lb, ub = detect_outliers_iqr(df, col)
    print(f'{col:35s} | Outliers: {count:3d} | Range: [{lb:.1f}, {ub:.1f}]')

## 7. Duplicate Records

In [ ]:
# Check for duplicate rows
duplicates = df.duplicated().sum()
print(f'Number of duplicate rows: {duplicates}')
print(f'Percentage of duplicates: {(duplicates / len(df)) * 100:.2f}%')

## 8. Key Insights Summary

In [ ]:
print('=' * 70)
print('KEY INSIGHTS FROM EXPLORATORY DATA ANALYSIS')
print('=' * 70)
print()
print('1. Dataset Shape: {} rows, {} columns'.format(df.shape[0], df.shape[1]))
print()
print('2. Target Variable (Biopsy) is highly imbalanced:')
print(f'   - Negative (0): {df["Biopsy"].value_counts()[0]} samples')
print(f'   - Positive (1): {df["Biopsy"].value_counts()[1]} samples')
print(f'   - Imbalance Ratio: {df["Biopsy"].value_counts(normalize=True)[1]*100:.2f}% positive')
print()
print('3. Missing Values: Several columns have missing values (encoded as "?"):')
print('   - STDs: Time since first diagnosis and STDs: Time since last diagnosis')
print('     have the most missing values')
print('   - Other columns have moderate missing values')
print()
print('4. Key Correlations with Biopsy:')
top_corr = target_corr.head(5)
for col, val in top_corr.items():
    print(f'   - {col}: {val:.3f}')
print()
print('5. Duplicate Records: {} duplicates found'.format(duplicates))
print()
print('6. Data Types: Mixed - need to convert object columns to numeric')
print()
print('7. Next Steps: Feature engineering, handling missing values,')
print('   handling class imbalance (SMOTE), and model training.')